## Calculating a Typical Meteorological Year

This notebook walks through the process of calculating a [Typical Meteorological Year](https://nsrdb.nrel.gov/data-sets/tmy), an hourly dataset used for applications in energy and building systems modeling. Because this represents average rather than extreme conditions, an TMY dataset is not suited for designing systems to meet the worst-case conditions occurring at a location. 

The TMY methodology here mirrors that of the Sandia/NREL TMY3 methodology, and uses historic and projected downscaled climate data available through the Cal-Adapt: Analytics Engine catalog. As this methodology heavily weights the solar radiation input data, be aware that the final selection of "typical" months may not be typical for other variables. 

Please note, the Analytics Engine also has an *Average Meteorological Year* functionality. The methods shown throughout this notebook will soon replace the underlying backend `climakitae` code for the AMY in order to better address our user needs, i.e., we are working to replace the AMY with the TMY methods. We provide this walkthrough to demonstrate confidence in the "AMY to TMY" conversion process for our users in the meantime. 

### Step 0: Set-up

Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [ ]:
import climakitae as ck
import pandas as pd
import xarray as xr
import numpy as np
from statsmodels.distributions.empirical_distribution import ECDF
import matplotlib.pyplot as plt
import pkg_resources
import holoviews
import datetime
# from datetime import date

# to calculate direct normal irradiance
!pip install pvlib
from pvlib import irradiance, solarposition
import pvlib

from climakitae.utils import get_closest_gridcell
from climakitae.derive_variables import _compute_dewpointtemp
from climakitae.unit_conversions import _convert_units

To use climakitae, load a new application:

In [ ]:
app = ck.Application()

### Step 1: Grab and process all required input data

The TMY3 method selects a "typical" month based on ten daily variables: max, min, and mean air and dew point temperatures, max and mean wind speed, global irradiance and direct irradiance ([NREL TMY3](https://www.nrel.gov/docs/fy08osti/43156.pdf)). Some of these variables are already available in the Analytics Engine data catalog, and some we will need to calculate ourselves.  

#### Step 1a: Select location of interest
TMYs are usually calculated for a specific location of interest, like a building or power plant (@Grace plase check me here). Here, we will use a known weather station location, via their latitude and longitude to extract the data that we need to calculate the TMY.  In the example below, we will look specifically at Los Angeles International Airport, but will note in the code below where you can provide your own location coordinates too. 

In [ ]:
# read in station file of CA HadISD stations
stn_file = pkg_resources.resource_filename("climakitae", "data/hadisd_stations.csv")
stn_file = pd.read_csv(stn_file, index_col=[0])
stn_file

In [ ]:
# grab los angeles airport
station_name = "Los Angeles International Airport"
one_station = stn_file.loc[stn_file['station'] == station_name]
stn_lat = one_station.LAT_Y.values
stn_lon = one_station.LON_X.values
stn_lat, stn_lon

#### Step 1b: Variables in catalog
When selecting data, there are several considerations to be aware of. The recommended minimum input of data is 15-20 years worth of daily data, and includes leap days (where available in our data). First, we will pre-load some data options to ensure that the same constraints are applied to every variable we retrieve from the catalog and calculate. For example, we will process all data for *Los Angeles County* at *45 km* over the 1990-2020 period. For data post-2014, we will utilize SSP 3-7.0, although scenario selection in the near-future is relatively independent. If calculating a TMY for the far-future, **carefully consider which scenario SSP to include**, as there will be **significant** differences present. 

In [ ]:
# default selections applicable to all variables selected
app.selections.data_type = "Gridded"
app.selections.area_average = "No"
app.selections.scenario_historical = ["Historical Climate"]
app.selections.scenario_ssp = ["SSP 3-7.0 -- Business as Usual"] # Important based on time period considered!!
app.selections.time_slice = (1990, 2020)
app.selections.timescale = "daily"
app.selections.resolution = "45 km"
app.selections.area_subset = "CA counties"
app.selections.cached_area = "Los Angeles County"

You may want to provide your own location instead of one of the HadISD stations above. If so, uncomment the cell below by removing the `#` symbol. To be able to extract the data variables from the Analytics Engine catalog, you will also need to at least provide the county your location resides in. If you know what the county is already, input the name below with the name capitalized (i.e., "Los Angeles County", "San Francisco County"). You do not have to "comment out" the app.selections.cached_area in the cell above from the default settings if you modify it in the cell below: it should update to your desired setting, but you can do so if you like. 

In [ ]:
## provide your own location, via latitude and longitude coordinates
# stn_lat = YOUR_LAT_HERE
# stn_lon = YOUR_LON_HERE
# app.selections.cached_area = "COUNTY_GOES_HERE"

Now that we have set up default settings, let's start retrieving data! First, retrieve min, max, and mean air temperature: 

In [ ]:
# max air temp
app.selections.variable = "Maximum air temperature at 2m"
app.selections.unit = "degC"
max_airtemp_data = app.retrieve()
max_airtemp_at_stn = get_closest_gridcell(max_airtemp_data, stn_lat, stn_lon) # retrieves data at lat-lon location

# min air temp
app.selections.variable = "Minimum air temperature at 2m"
app.selections.unit = "degC"
min_airtemp_data = app.retrieve()
min_airtemp_at_stn = get_closest_gridcell(min_airtemp_data, stn_lat, stn_lon) # retrieves data at lat-lon location

# mean air temperature
app.selections.variable = "Air Temperature at 2m" 
app.selections.unit = "degC"
mean_airtemp_data = app.retrieve()
mean_airtemp_data.name = "Mean air temperature at 2m" # rename for clarity
mean_airtemp_at_stn = get_closest_gridcell(mean_airtemp_data, stn_lat, stn_lon) # retrieves data at lat-lon location

Retrieve max and mean wind speed:

In [ ]:
# max wind speed
app.selections.variable = "Maximum wind speed at 10m"
app.selections.unit = "m s-1"
max_windspd_data = app.retrieve()
max_windspd_at_stn = get_closest_gridcell(max_windspd_data, stn_lat, stn_lon) # retrieves data at lat-lon location

# mean wind speed
app.selections.variable = "Mean wind speed at 10m"
app.selections.unit = "m s-1"
mean_windspd_data = app.retrieve()
mean_windspd_at_stn = get_closest_gridcell(mean_windspd_data, stn_lat, stn_lon) # retrieves data at lat-lon location

#### Step 1c: Variables that need to be calculated
Next, we will need to retrieve **hourly** data from the catalog to calculate the following variables, as they are not natively within the Analytics Engine data catalog. 
- Max, min, and mean dew point temperature
- Direct normal irradiance
- Global horizontal irradiance

In [ ]:
# switch to hourly timescale
app.selections.timescale = "hourly"

Next, calculate max, min, and mean dew point temperature:

In [ ]:
# dew point temperature
app.selections.variable = "Dew point temperature"
app.selections.unit = "degC"
dewpt_data = app.retrieve()
dewpt_data_at_stn = get_closest_gridcell(dewpt_data, stn_lat, stn_lon) # retrieves data at lat-lon location

In [ ]:
max_dewpt_data_at_stn = dewpt_data_at_stn.resample(time="1D").max() # daily max dewpoint temp
max_dewpt_data_at_stn.name = "Daily max dewpoint temperature" # rename for clarity

min_dewpt_data_at_stn = dewpt_data_at_stn.resample(time="1D").min() # daily min dewpoint temp
min_dewpt_data_at_stn.name = "Daily min dewpoint temperature" # rename for clarity

mean_dewpt_data_at_stn = dewpt_data_at_stn.resample(time="1D").mean() # daily mean dewpoint temp
mean_dewpt_data_at_stn.name = "Daily mean dewpoint temperature" # rename for clarity

Next, retrieve global horizontal irradiance. GHI is within the Analytics Enginge catalog at daily resolutions, but for the TMY methodology, we need to calculate the total accumulated GHI received over the course of the day, so we will retrieve hourly data instead and calculate the necessary information below. 

In [ ]:
# global irradiance
app.selections.variable = "Instantaneous downwelling shortwave flux at bottom"
global_irradiance_data = app.retrieve()
global_irradiance_data.name = "Global horizontal irradiance" # rename for clarity
ghi_at_stn = get_closest_gridcell(global_irradiance_data, stn_lat, stn_lon) # retrieves data at lat-lon location
total_ghi_at_stn = ghi_at_stn.resample(time="1D").sum() # total global horizontal irradiance (accumulation of hourly data over a 24-hour period)

In [ ]:
## ONCE IN CATALOG (AND REMOVE DNI CALCULATION BELOW)

# # direct normal irradiance
# app.selections.variable = "VARIABLE_NAME_HERE"
# direct_irradiance_data = app.retrieve()
# direct_irradiance_data.name = "Direct normal irradiance" # rename for clarity
# dni_at_stn = get_closest_gridcell(direct_irradiance_data, stn_lat, stn_lon) # retrieves data at lat-lon location
# total_dni_at_stn = dni_at_stn.resample(time="1D").sum() # total direct normal irradiance (accumulation of hourly data over a 24-hour period)

Lastly, we will calculate **direct normal irradiance**. As direct normal irradiance is not presently in the Analytics Engine catalog (but is coming soon!), we will use the DIRINT method with the dewpoint temperature enhancement ([Perez et al. 1992](https://www.researchgate.net/publication/279868352_Dynamic_global-to-direct_irradiance_conversion_models)) to calculate, which uses the **global horizontal irradiance** and the solar zenith angle. 

Once direct normal irradiance is in the Analytics Engine data catalog, we will update this notebook to reflect the changes as well. 

**notes**
- *Perez, R., P. Ineichen, E. Maxwell, R. Seals and A. Zelenka, (1992). “Dynamic Global-to-Direct Irradiance Conversion Models”. ASHRAE Transactions-Research Series, pp. 354-369*
- *Different methods: https://assessingsolar.org/notebooks/decomposition_models.html*

First calculate the solar zenith angle: 

In [ ]:
# calculate solar position, including solar zenith angle
solpos = solarposition.get_solarposition(time = pd.date_range('1990-01-01', '2021-01-01', freq='H'), # need to avoid hardcoding dates here... in progress
                                        latitude = stn_lat,
                                        longitude = stn_lat)
solpos

Next, we will set-up a helper function `dni_by_sim` to calculate direct normal irradiance. 

In [ ]:
def dni_by_sim(ghi_data, dew_data, solpos):
    """Calculates the direct normal irradiance per simulation, with DIRINT method and dew point improvements"""
    df = pd.DataFrame()
    for sim in range(4):
        df_by_sim = ghi_data.squeeze().isel(simulation=sim).to_dataframe()
        dt_by_sim = dew_data.squeeze().isel(simulation=sim).to_dataframe()
        dirint_by_sim = pvlib.irradiance.dirint(ghi=df_by_sim['Global horizontal irradiance'],
                                                temp_dew=dt_by_sim['Dew point temperature'],
                                                solar_zenith=solpos['apparent_zenith'],
                                                times=solpos.index)
        
        df[df_by_sim.simulation.values[0]] = dirint_by_sim      
    return df

Calculate direct normal irradiance and check out the values next:

In [ ]:
# need to adjust for differences in calendar (leap days)

dni = dni_by_sim(ghi_at_stn, dewpt_data, solpos) # hourly direct normal irradiance
total_dni_at_stn = dni.resample(time="1D").sum() # total direct normal irradiance (accumulation of hourly data over a 24-hour period)
total_dni_at_stn

#### Step 1d: Load all variables
Now that we have all of our data retrieved and calculated, it is time to actually load the data into memory. Previously, xarray has lazily loaded the data, but we will actually grab it now. 

In [ ]:
all_vars = xr.merge([max_airtemp_at_stn.squeeze(), min_airtemp_at_stn.squeeze(), mean_airtemp_at_stn.squeeze(),
                     max_dewpt_data_at_stn.squeeze(), min_dewpt_data_at_stn.squeeze(), mean_dewpt_data_at_stn.squeeze(),
                     max_windspd_at_stn.squeeze(), mean_windspd_at_stn.squeeze(),
                     total_ghi_at_stn.squeeze()]) #, total_dni_at_stn])

all_vars

In [ ]:
# load all indices in
all_vars = all_vars.compute() # app.load does not work here because of uneven chunk size on time dimension

In [ ]:
all_vars

### Step 2: Calculate cumultative distribution functions

For the TMY, the cumulative distribution function gives the proportion of values that are less than or equal to a specified value of the index. In this case, we want to identify months that are as close to the long-term climatology for each variable. 

#### Step 2a: Calculate long-term climatology CDFs for each index
First, we need to calculate the long-term climatology for each index for each month so we can establish the baseline pattern. 

In [ ]:
def get_cdf(data):
    """Calculate the Cumulative Distribution Function (CDF) for the given property"""
    
    data_sim = data.squeeze().isel(simulation=0)
    count, bins_count = np.histogram(data_sim, 
                                     bins = np.linspace(data_sim.min().values, data_sim.max().values, 1024))
    cdf = np.cumsum(count/sum(count))
    df = pd.DataFrame(data = cdf, columns = [str(data_sim.simulation.values)])
    
    for i in range(1, 4):
        data_sim = data.squeeze().isel(simulation=i)
        count, bins_count = np.histogram(data_sim,
                                         bins = np.linspace(data_sim.min().values, data_sim.max().values, 1024))
        cdf = np.cumsum(count/sum(count))        
        df[str(data_sim.simulation.values)] = cdf
    return df
        
        # plt.plot(bins_count[1:], cdf, label=data_sim.simulation.values)
        # plt.legend()
        
def cdf_by_month(y):
    return y.groupby('time.month').map(get_cdf)

In [ ]:
# long-term climatology CDF goes here
cdf_climatology = get_cdf(all_vars)

Let's take a look at what the climatological pattern looks like for maximum air temperature now: 

In [ ]:
get_cdf(all_vars['Maximum air temperature at 2m']).hvplot.line() # x axis is not quite right - needs to be the data unit range, not the bins

In [ ]:
# produce subplots that illustrates each climatological CDF with variable name provided, with month as a toggle selection

#### Step 2b: Calculate CDFs for each index for all months

Next, we will calculate CDF for each month and each variable, for which we ultimatley will compare against climatology. For the individual months, we must also exclude leap year days and the period of time during two major volcanic eruptions (El Chichón, May 1982 until December 1984; and Pinatubo, June 1991 to December 1994) as the aerosols have an impact on solar variables. We have several handy helper functions to exlude this data from our data next. 

In [ ]:
def remove_leapdays(data):
    """Drops Feb 29th in leap years"""
    leap_m = np.logical_and(data.time.dt.is_leap_year, data.time.dt.dayofyear==60) # identifies leap years
    return data.where(~leap_m, drop=True) # drops feb 29 only if leap year

def remove_volcano(data):
    """Excludes specific months from CDF calculation due to volcanic eruptions"""
    elchichon = data.sel(time=slice('1982-05-01', '1984-12-31'))
    pinatubo = data.sel(time=slice('1991-06-01', '1994-12-31'))
    
    data = data.drop_sel(time=elchichon) # el chichón: 5/1/1982 - 12/31/1984    
    data = data.drop_sel(time=pinatubo) # pinatubo: 6/1/1991 - 12/31/1994
    
    # only working so far on dataarrays, not datasets ugh
                                      
    return data

In [ ]:
all_vars_monthly = remove_leapdays(all_vars)
all_vars_monthly = remove_volcano(all_vars_monthly)

all_vars_monthly

Calculate the monthly CDF next, using the modified (leap and volcano removed) data.

In [ ]:
# needs monthly groupby here next
cdf_monthly = get_cdf(all_vars_monthly) # something

In [ ]:
get_cdf(all_vars['Maximum air temperature at 2m']).hvplot.line()

In [ ]:
# would be great if we could have the climatology lines "bolded" and then provide each individual month per simulation as thin lines to show full range??

### Step 3: Compare climatology CDF to monthly CDF for each variable

#### Step 3a: Calculate the Finkelstein-Schafer statistic 
The Finkelstein-Schafer statistic determines the absolute difference between the long-term climatology and candidate CDF profiles, and considers the number of days within each month ([Finkelstein and Schafer 1971](https://academic.oup.com/biomet/article-abstract/58/3/641/233677)). 

In [ ]:
def fs_statistic(cdf_climatology, cdf_month):
    """
    Calculates the Finkelstein-Schafer statistic:
    Absolute difference between long-term climatology and candidate CDF, divided by number of days in month
    """
    len_month = number_of_days_in_month
    da_fs = (cdf_climatology - cdf_month).abs().mean()
    return da_fs / len_month

In [ ]:
all_vars_fs = fs_statistic(cdf_climatology, cdf_monthly)

#### Step 3b: Weight the F-S statistic

Next, we weight the F-S statistic results based on the input variables. The [TMY3](https://www.nrel.gov/docs/fy08osti/43156.pdfhttps://www.nrel.gov/docs/fy08osti/43156.pdf) method places a higher weight to the solar variables (global irradiance and direct irradiance), which we follow here. 

Respectively, the maximum and minimum air temperatures and dewpoint temperatures, and maximum and mean wind speed are weighted at 1/20 each. Mean air and dewpoint temperatures are weighted at 2/20 each. Finally, global and direct irradiance each have a weight of 5/20. In the next cell, we provide a function that calculates this weight for us. 
WS = Weight * F-S value

In [ ]:
def weighted_fs(da_fs):
    """Weights the F-S statistics based on TMY3 methodology"""
    
    
    da_fs.max_airtemp * (1/20)
    
    
    return da_fs_weighted

#### Step 3c: Select candidate months for consideration
Once weighted, we select the top 5 candidate months that have lowest weighted sums, meaning that these 5 months are the closest to the long-term climatology for that month. 

In [ ]:
# for each month, rank all available years, lowest to highest
# select first five as "candidates"

candidate_months = sorted(weighted_fs(DATA))[:5]

#### Step 3d: Rank candidate months for selection
Rank with respect to closeness of month to long-term mean and median

"Typical" month selected by choosing among 5 months with lowest WS values the month with the smallest deviation from the long-term CDF

In [ ]:
# using 5 candidate months
# identify which month is closest to climatology



### Step 4: Consider persistence of weather patterns

#### Step 4a: Monthly mean and median, persistence of wx patterns
- Mean daily dry bulb temp
- Global horizontal radiation
- frequency and run length above 67th percentile/below 33rd percentile

Exclude month with longest run, month with most runs, month with zero runs

***Noting that https://github.com/TerriaJS/aremi-tmy/blob/master/sandia/tmy.py#L118 skips this step***

### Step 5: Concatenate months together

Months concatenated together and discontinuities at month interface are smoothed for 6 hours on either side using curve-fitting techniques

In [ ]:
# provide a list/table/output of which month comes from which year

### Step 6: Generate the TMY data outputs

Generally, the following is outputted using the TMY months:
- Date & time (UTC)
- T2m [°C] - Dry bulb (air) temperature.
- RH [%] - Relative Humidity.
- G(h) [W/m2] - Global horizontal irradiance.
- Gb(n) [W/m2] - Direct (beam) irradiance -- ***currently not available to calculate in AE -- but can be calculated here and added in as derived variable?***
- Gd(h) [W/m2] - Diffuse horizontal irradiance.
- IR(h) [W/m2] - Infrared radiation downwards -- ***Instantaneous downwelling longwave flux at bottom -- check that this is what we want here***
- WS10m [m/s] - Windspeed.
- WD10m [°] - Wind direction.
- SP [Pa] - Surface (air) pressure.

In [ ]:
# step 1 - using TMY selction of months
# step 2 - grab available variables at hourly timescales as dataarrays
# step 3 - concat variable dataarrays into pandas df
# step 4 - visualize (head) table
# step 5 - export table ==> this is the "TMY data file"